## Prototyping the Logic

Before writing the dbt model, we can prototype the logic in this notebook using the local DuckDB database.

Let's create a logic that categorizes customers based on their email domains—a task that involves string manipulation which is often cleaner in Python.

In [ ]:
import duckdb
import pandas as pd

# Connect to the local dbt DuckDB database
# Note: Ensure no other process (like dbt run) is holding a lock on the DB file
con = duckdb.connect("my_database.duckdb")

# Load data (Simulating dbt.ref('dim_customers'))
# We read from the 'analytics' schema where dbt materialized the models
df = con.sql("SELECT * FROM analytics.dim_customers").df()

# Preview the data
print(f"Loaded {len(df)} customers")
df[["customer_id", "email"]].head()

In [ ]:
# Define our Python transformation logic
def categorize_email(email):
    if not email:
        return "Unknown"
    domain = email.split("@")[1]
    if "gmail" in domain:
        return "Gmail"
    elif "yahoo" in domain:
        return "Yahoo"
    elif "hotmail" in domain or "outlook" in domain:
        return "Microsoft"
    else:
        return "Other"


# Apply the logic using pandas
df["email_provider"] = df["email"].apply(categorize_email)

# Analyze the results
summary = df["email_provider"].value_counts()
print("Email Provider Distribution:")
print(summary)

df[["email", "email_provider"]].head()

## Creating the dbt Model

Now that we know our logic works, we can wrap it in a dbt model file.

**File:** `models/marts/customers_enriched_python.py`

```python
import pandas as pd

def model(dbt, session):
    # Configure the model
    dbt.config(
        materialized='table',
        packages=['pandas'] # List requirements here if needed (mostly for Snowflake/BigQuery)
    )

    # Reference the upstream model
    # dbt.ref returns a DuckDBPyRelation in dbt-duckdb
    dim_customers = dbt.ref('dim_customers')

    # Convert to pandas DataFrame to use pandas methods
    pdf = dim_customers.df()

    # Define the logic (same as above)
    def categorize_email(email):
        if not email:
            return 'Unknown'
        try:
            domain = email.split('@')[1]
            if 'gmail' in domain:
                return 'Gmail'
            elif 'yahoo' in domain:
                return 'Yahoo'
            elif 'hotmail' in domain or 'outlook' in domain:
                return 'Microsoft'
            else:
                return 'Other'
        except:
            return 'Invalid'

    # Apply the transformation
    pdf['email_provider'] = pdf['email'].apply(categorize_email)

    # Return the result
    # dbt will handle writing this back to the database
    return pdf
```

## Running the Model

To build this model, you would run:

```bash
dbt run -s customers_enriched_python
```